In [ ]:
import wfdb
import ast
import pandas as pd
import numpy as np
import neurokit2 as nk
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer
from tqdm import tqdm

# Configuration
DATA_PATH = 'data/ptbxl/' 
SAMPLING_RATE = 100
SAMPLE = 200 

print("Environment Ready.")

Environment Ready.


In [ ]:
# Load the manifest 
df = pd.read_csv(os.path.join(DATA_PATH, 'ptbxl_database.csv'), index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))

# Pick the first n records to download
records_to_get = df.head(SAMPLE).filename_lr.tolist()

print(f"Downloading {len(records_to_get)} records to {DATA_PATH}...")

# Download from PhysioNet
wfdb.dl_database('ptb-xl', 
                 dl_dir=DATA_PATH, 
                 records=records_to_get, 
                 annotators=None)

print(f"Download Complete! You now have the signals for the first {SAMPLE} patients.")

In [ ]:
# Cleans signal, finds R-peaks, and extracts HRV features using NeuroKit2.
def extract_ecg_features(file_path):

    try:
        # Load the signal (Lead I)
        full_path = os.path.join(DATA_PATH, file_path)
        signal, meta = wfdb.rdsamp(full_path)
        lead_1 = signal[:, 0]
        
        signals, info = nk.ecg_process(lead_1, sampling_rate=SAMPLING_RATE)
        
        # Get HRV and interval metrics
        results = nk.ecg_intervalrelated(signals)
        return results
    except Exception as e:
        # Skip records that are too noisy or corrupted
        return None

print("Feature extraction function defined.")

In [ ]:
features_list = []
labels = []

print("Processing ECG signals with data cleaning...")

# Extract the Data
for idx, row in tqdm(df.head(SAMPLE).iterrows(), total=SAMPLE):
    f_data = extract_ecg_features(row.filename_lr)
    
    if f_data is not None:
        clean_row = f_data.iloc[0].apply(lambda x: x.item() if hasattr(x, 'item') else x)
        
        features_list.append(clean_row)
        labels.append(0 if 'NORM' in row.scp_codes else 1)

if len(features_list) > 0:
    # Build DataFrame
    X = pd.concat(features_list, axis=1).T.reset_index(drop=True)
    
    # Force everything to be a float to allow mean to work 
    X = X.apply(pd.to_numeric, errors='coerce')
    
    X = X.fillna(X.mean())
    
    y = np.array(labels)
    print(f"Dataset built: {X.shape[0]} samples, {X.shape[1]} features.")
else:
    print("Error: No features were extracted.")

In [ ]:
# Replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Robust Scaling Pipeline
imputer = SimpleImputer(strategy='median') 
X_train_clean = imputer.fit_transform(X_train)
X_test_clean = imputer.transform(X_test)

# Scale features 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

# Initialize and Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Predict and Report
y_pred = model.predict(X_test_scaled)

print("\nFINAL CRELY PIPELINE REPORT\n")

# Check if enough samples to show both classes
unique_labels = np.unique(np.concatenate([y_test, y_pred]))
target_names = ['Normal', 'Abnormal']
# Filter target names based on which labels are actually present in the test/pred set
present_names = [target_names[i] for i in unique_labels]

print(classification_report(y_test, y_pred, target_names=present_names))

/Users/aparikh27/Desktop/Crely/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['HRV_SDANN1' 'HRV_SDNNI1' 'HRV_SDANN2' 'HRV_SDNNI2' 'HRV_SDANN5'
 'HRV_SDNNI5' 'HRV_ULF' 'HRV_VLF' 'HRV_LF' 'HRV_HF' 'HRV_VHF' 'HRV_LFHF'
 'HRV_LFn' 'HRV_HFn' 'HRV_LnHF' 'HRV_DFA_alpha2']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/aparikh27/Desktop/Crely/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['HRV_SDANN1' 'HRV_SDNNI1' 'HRV_SDANN2' 'HRV_SDNNI2' 'HRV_SDANN5'
 'HRV_SDNNI5' 'HRV_ULF' 'HRV_VLF' 'HRV_LF' 'HRV_HF' 'HRV_VHF' 'HRV_LFHF'
 'HRV_LFn' 'HRV_HFn' 'HRV_LnHF' 'HRV_DFA_alpha2']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



FINAL CRELY PIPELINE REPORT

              precision    recall  f1-score   support

      Normal       0.71      0.92      0.80        26
    Abnormal       0.67      0.29      0.40        14

    accuracy                           0.70        40
   macro avg       0.69      0.60      0.60        40
weighted avg       0.69      0.70      0.66        40

